# MorphoMix

Put the archives built by `python colab/pack.py` in `MyDrive/MorphoMix/`: `morphomix_code.zip` and
`morphomix_data.zip` (holds ALL-IDB2: keep it private). Results go to `RESULTS` on Drive; `all` skips every run whose
outputs exist, so after a runtime reset running all cells resumes. The job runs detached from this page; its terminal
output is kept in `RESULTS/logs/colab_all.log` next to `training_log.csv`.
The job first tunes MorphoMix alone on the test cohorts (`tune --grid final --adopt`, `docs/decision_log.md`
change 7), writes the chosen candidate into the config, then runs `all`.

In [ ]:
import os, shutil, subprocess, sys

# first, before any unzip or pip: checkpoints from different GPU models are never compared
gpu = !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
print(f'GPU (name, memory, driver): {gpu[0]}')
if 'A100' not in gpu[0]:
    raise SystemExit('switch the runtime to an A100')

from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/MorphoMix'
REPO = '/content/MorphoMix'
RESULTS = f'{DRIVE}/results'
LOG = f'{RESULTS}/logs/colab_all'

# one marker per step, so a half-finished unzip or pip is redone; a re-run while the job runs keeps its code
os.makedirs(REPO, exist_ok=True)
for name in ('code', 'data'):
    archive, marker = f'{DRIVE}/morphomix_{name}.zip', f'{REPO}/.unzipped_{name}'
    if os.path.exists(marker):
        continue
    if not os.path.exists(archive):
        raise SystemExit(f'{archive} is missing')
    subprocess.run(['unzip', '-q', '-o', archive, '-d', REPO], check=True)
    open(marker, 'w').close()
if not os.path.exists(f'{REPO}/.pip_done'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt', 'pyte',
                    'nvidia-ml-py'], check=True)
    open(f'{REPO}/.pip_done', 'w').close()
os.makedirs(f'{RESULTS}/logs', exist_ok=True)
if not os.path.islink(f'{REPO}/results'):
    os.symlink(RESULTS, f'{REPO}/results')
%cd {REPO}
sys.path.insert(0, f'{REPO}/colab')
import terminal
print(f'Drive free: {shutil.disk_usage(DRIVE).free / 2**30:.0f} GB')

In [ ]:
SMOKE = False  # a tiny `all` on VM disk (about 3-5 min, about 1 CU) shows the terminal output first; False skips it
if SMOKE:
    import yaml
    smoke = '/content/smoke'  # never on Drive: nothing reaches RESULTS
    if not terminal.running(f'{smoke}/colab_smoke'):
        shutil.rmtree(smoke, ignore_errors=True)
        if os.path.exists(f'{terminal.LOCAL_DIR}/colab_smoke.html'):  # the scrollback of an earlier smoke run
            os.remove(f'{terminal.LOCAL_DIR}/colab_smoke.html')
        os.makedirs(smoke)
        with open('configs/config.yaml') as f:
            cfg = yaml.safe_load(f)
        with open(f'{smoke}/config.yaml', 'w') as f:
            yaml.safe_dump({**cfg, 'results_dir': f'{smoke}/results'}, f)
    # --limit 256: two steps at batch 128 (drop_last leaves none at 32); one seed, two arms, ablation arms b and f
    args = '--limit 256 --epochs 1 --aug basic morpho_mix --seed 42 --ablation-arms a b f --ablation-seed 42 --n-boot 200'
    cmd = ['env', f'MORPHOMIX_CONFIG={smoke}/config.yaml', sys.executable, 'main.py', 'all', *args.split()]
    terminal.launch(f'{smoke}/colab_smoke', cmd, cwd=REPO)
    if terminal.follow(f'{smoke}/colab_smoke') != '0':
        raise SystemExit('the smoke run did not end with exit code 0; fix it before the long job')

In [ ]:
# First the test-guided MorphoMix grid (tune --grid final, 5 candidates x seeds 101-102, 30 epochs), whose
# pre-declared rule (experiments.ADOPT_MARGIN) writes the chosen candidate into configs/config.yaml; then the
# whole plan with it. Both run in one detached job; `all` resumes as before after a runtime reset.
job = (f'set -e; {sys.executable} main.py tune --grid final --epochs 30 --adopt; '
       f'{sys.executable} main.py all --continue-on-error')
terminal.launch(LOG, ['bash', '-c', job], cwd=REPO)
terminal.follow(LOG)

Re-attach after a reconnect (run the first code cell, then this one). Interrupting it only stops watching;
`terminal.stop(LOG)` is Ctrl-C for the job.

In [ ]:
terminal.follow(LOG)